# Scenario Comparison Analysis

This notebook provides an interactive GUI for comparing two simulation scenarios side-by-side.

**Note:** Both scenarios being compared share the same topology and ISP subgraphs.
The comparison shows how different routing strategies (e.g., disaster-aware vs. standard routing) affect the same network.

## Features:

1. **Scenario Selection**: Load and compare any two scenarios from the output folder
2. **View Modes**:
   - **Topology View**: Compare ISP subgraphs in normal/disaster mode
   - **Statistics View**: Compare metrics with filtering and multiple visualization types
3. **Filtering Options**: No Filter, Migration Traffic Only, Exclude Migration Traffic, By ISP, By Node
4. **Visualization Types**: Blocking Probability, Availability, Link Utilization, Network Usage
5. **Dynamic Controls**: Time bucket slider for temporal analyses, Top N Links slider for link utilization (auto-adjusts to total link count)
6. **Timing Markers**: ISP Migration Start Times and Disaster Period
7. **Smart Caching**: Plots are cached automatically - switching between views or scenarios without changing parameters reuses cached plots for faster navigation

## 1. Setup and Imports

In [1]:
# Batch comparison utilities
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

def group_scenarios_by_directory(scenario_names, scenario_paths):
    """
    Group scenarios by their parent directory.
    
    Args:
        scenario_names: List of scenario full paths
        scenario_paths: Dict mapping scenario_name -> {'pkl': path, 'csv': path}
    
    Returns:
        Dict mapping directory_path -> list of scenario names in that directory
    """
    groups = defaultdict(list)
    
    for name in scenario_names:
        # Get the directory path (parent of the pkl file)
        pkl_path = scenario_paths[name]['pkl']
        dir_path = str(pkl_path.parent.relative_to(Path("output")))
        
        # Use directory path as key, or "root" for direct files
        if dir_path == ".":
            dir_key = "root"
        else:
            dir_key = dir_path.replace("\\", "/")
        
        groups[dir_key].append(name)
    
    return dict(groups)


def average_dataframes(dataframes_list, bucket_size=1.0):
    """
    Average multiple dataframes by computing metrics per time bucket.
    
    For time-series data, this averages blocking rates, availability, etc. per time bucket.
    For link utilization, it averages utilization values per link.
    
    Args:
        dataframes_list: List of dataframes to average
        bucket_size: Time bucket size for temporal averaging
    
    Returns:
        Averaged dataframe with same structure as input
    """
    if not dataframes_list:
        return pd.DataFrame()
    
    if len(dataframes_list) == 1:
        return dataframes_list[0].copy()
    
    # Combine all dataframes
    all_dfs = []
    for df in dataframes_list:
        if len(df) > 0:
            all_dfs.append(df.copy())
    
    if not all_dfs:
        return pd.DataFrame()
    
    # Get time range
    min_time = min(df['tempo_criacao'].min() for df in all_dfs if len(df) > 0)
    max_time = max(df['tempo_criacao'].max() for df in all_dfs if len(df) > 0)
    
    # Create time buckets
    bins = np.arange(min_time, max_time + bucket_size, bucket_size)
    bucket_centers = [(bins[i] + bins[i+1]) / 2 for i in range(len(bins) - 1)]
    
    # Calculate blocking probability per bucket for each dataframe
    blocking_rates_per_df = []
    availability_rates_per_df = []
    counts_per_df = []
    
    for df in all_dfs:
        blocking_rates = []
        availability_rates = []
        counts = []
        
        for i in range(len(bins) - 1):
            bucket_df = df[
                (df['tempo_criacao'] >= bins[i]) & 
                (df['tempo_criacao'] < bins[i + 1])
            ]
            if len(bucket_df) > 0:
                blocked = bucket_df['bloqueada'].sum()
                total = len(bucket_df)
                blocking_rate = blocked / total
                availability_rate = 1 - blocking_rate
            else:
                blocking_rate = np.nan
                availability_rate = np.nan
                total = 0
            
            blocking_rates.append(blocking_rate)
            availability_rates.append(availability_rate)
            counts.append(total)
        
        blocking_rates_per_df.append(blocking_rates)
        availability_rates_per_df.append(availability_rates)
        counts_per_df.append(counts)
    
    # Average across dataframes (weighted by count)
    avg_blocking_rates = []
    avg_availability_rates = []
    avg_counts = []
    
    for i in range(len(bucket_centers)):
        # Get values for this bucket across all dataframes
        bucket_blocking = [rates[i] for rates in blocking_rates_per_df if not np.isnan(rates[i])]
        bucket_availability = [rates[i] for rates in availability_rates_per_df if not np.isnan(rates[i])]
        bucket_counts = [counts[i] for counts in counts_per_df]
        
        if bucket_blocking:
            # Weighted average by count
            total_count = sum(bucket_counts)
            if total_count > 0:
                # For blocking rate, we need to reconstruct from counts
                total_blocked = sum(
                    blocking_rates_per_df[j][i] * counts_per_df[j][i] 
                    for j in range(len(all_dfs)) 
                    if not np.isnan(blocking_rates_per_df[j][i])
                )
                avg_blocking = total_blocked / total_count if total_count > 0 else 0
                avg_availability = 1 - avg_blocking
            else:
                avg_blocking = np.nan
                avg_availability = np.nan
        else:
            avg_blocking = np.nan
            avg_availability = np.nan
        
        avg_blocking_rates.append(avg_blocking)
        avg_availability_rates.append(avg_availability)
        avg_counts.append(sum(bucket_counts))
    
    # Create averaged dataframe structure
    # We'll create a synthetic dataframe with representative rows
    # For simplicity, we'll use the first dataframe as template and adjust values
    template_df = all_dfs[0].copy()
    
    # Create averaged rows based on bucket centers
    averaged_rows = []
    for i, bucket_center in enumerate(bucket_centers):
        if not np.isnan(avg_blocking_rates[i]) and avg_counts[i] > 0:
            # Create representative rows for this bucket
            num_rows = max(1, int(avg_counts[i] / len(all_dfs)))  # Approximate rows per bucket
            
            for _ in range(num_rows):
                row = template_df.iloc[0].copy()
                row['tempo_criacao'] = bucket_center
                row['bloqueada'] = np.random.random() < avg_blocking_rates[i]
                row['requisicao_de_migracao'] = template_df['requisicao_de_migracao'].iloc[0] if len(template_df) > 0 else False
                averaged_rows.append(row)
    
    if averaged_rows:
        averaged_df = pd.DataFrame(averaged_rows)
        # Recalculate blocking to match average
        # For a more accurate representation, we'll directly set blocking rates
        return averaged_df
    else:
        # Fallback: return first dataframe
        return all_dfs[0].copy()


def average_dataframes_simple(dataframes_list):
    """
    Simple averaging: concatenate and average blocking rates per time bucket.
    This is a more straightforward approach that preserves all data.
    
    Args:
        dataframes_list: List of dataframes to average
    
    Returns:
        Concatenated dataframe (for now, we'll use this for averaging metrics)
    """
    if not dataframes_list:
        return pd.DataFrame()
    
    if len(dataframes_list) == 1:
        return dataframes_list[0].copy()
    
    # For averaging, we'll concatenate and let the visualization functions
    # handle the averaging by grouping scenarios with the same name
    # This is simpler and more accurate
    return pd.concat(dataframes_list, ignore_index=True)


def create_averaged_scenario_data(scenarios_dict, dataframes_dict):
    """
    Create averaged scenario data by grouping scenarios in the same directory.
    
    Args:
        scenarios_dict: Dict mapping scenario_name -> scenario object
        dataframes_dict: Dict mapping scenario_name -> dataframe
    
    Returns:
        Tuple of (averaged_scenarios_dict, averaged_dataframes_dict, group_info)
        where group_info maps averaged_name -> list of original scenario names
    """
    # Group scenarios by directory
    scenario_paths = {}
    for name in scenarios_dict.keys():
        # Extract directory from name
        if '/' in name:
            dir_path = '/'.join(name.split('/')[:-1])
            scenario_paths[name] = dir_path
        else:
            scenario_paths[name] = "root"
    
    # Group by directory
    groups = defaultdict(list)
    for name, dir_path in scenario_paths.items():
        groups[dir_path].append(name)
    
    # Create averaged scenarios
    averaged_scenarios = {}
    averaged_dataframes = {}
    group_info = {}
    
    for dir_path, scenario_names in groups.items():
        if len(scenario_names) == 1:
            # Single scenario, no averaging needed
            name = scenario_names[0]
            averaged_name = dir_path if dir_path != "root" else name
            averaged_scenarios[averaged_name] = scenarios_dict[name]
            averaged_dataframes[averaged_name] = dataframes_dict[name]
            group_info[averaged_name] = [name]
        else:
            # Multiple scenarios, average them
            averaged_name = dir_path if dir_path != "root" else f"averaged_{scenario_names[0]}"
            
            # Use first scenario as representative (they should have same config)
            averaged_scenarios[averaged_name] = scenarios_dict[scenario_names[0]]
            
            # Average dataframes
            dfs_to_average = [dataframes_dict[name] for name in scenario_names]
            averaged_dataframes[averaged_name] = average_dataframes_simple(dfs_to_average)
            
            group_info[averaged_name] = scenario_names
    
    return averaged_scenarios, averaged_dataframes, group_info

def detect_batch_directories(output_path):
    """
    Detect directories that contain multiple scenarios (batch directories).
    
    Returns:
        dict mapping directory_path -> list of scenario file paths in that directory
    """
    batch_dirs = defaultdict(list)
    
    # Find all scenario files in subdirectories
    for pkl_file in output_path.rglob("scenario_*.pkl"):
        dir_path = str(pkl_file.parent.relative_to(output_path)).replace("\\", "/")
        csv_file = pkl_file.parent / f"dataframe_{pkl_file.stem.replace('scenario_', '')}.csv"
        
        if csv_file.exists():
            batch_dirs[dir_path].append({
                'pkl': pkl_file,
                'csv': csv_file,
                'name': pkl_file.stem.replace('scenario_', '')
            })
    
    # Return only directories with multiple scenarios
    return {
        dir_path: files 
        for dir_path, files in batch_dirs.items() 
        if len(files) > 1
    }

print("✓ Batch comparison utilities loaded!")


✓ Batch comparison utilities loaded!


In [2]:
from pathlib import Path
import ast

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# Enable autoreload for module changes
%load_ext autoreload
%autoreload 2

# Import analysis and visualization functions from modules
from simulador.analysis.scenario_comparison import (
    load_scenario_pair,
    apply_filter,
    create_comparison_data,
    create_multi_scenario_comparison_data
)

from simulador.visualization.comparison_plots import (
    plot_blocking_probability_comparison,
    plot_availability_comparison,
    plot_link_utilization_comparison,
    plot_network_usage_comparison,
    visualize_isp_topology_comparison,
    visualize_isp_topology,
    plot_blocking_probability_multi,
    plot_availability_multi,
    plot_link_utilization_multi,
    plot_network_usage_multi
)

from simulador.visualization.comparison_table import (
    create_comparison_table_widget,
    create_multi_scenario_table_widget
)

import pickle

# Style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 8)

print("✓ All modules loaded successfully!")

✓ All modules loaded successfully!


In [3]:
# Updated load_scenarios function that handles batch directories
# This function should replace the load_scenarios function in create_comparison_gui()

def load_scenarios_with_batch_support(selected_names, scenario_paths, batch_directories, scenario_display_names):
    """
    Load scenarios, automatically averaging batch directories.
    
    Args:
        selected_names: List of selected scenario names or directory paths
        scenario_paths: Dict mapping scenario_name -> {'pkl': path, 'csv': path}
        batch_directories: Dict mapping directory_path -> list of scenario file info
        scenario_display_names: Dict mapping full_path -> display_name
    
    Returns:
        dict with 'scenarios' and 'dataframes' keys, or None on error
    """
    scenarios = {}
    dataframes = {}
    
    for name in selected_names:
        # Check if this is a batch directory
        if name in batch_directories:
            # Load and average all scenarios in this directory
            dir_scenarios = []
            dir_dataframes = []
            
            for file_info in batch_directories[name]:
                with open(file_info['pkl'], "rb") as f:
                    dir_scenarios.append(pickle.load(f))
                dir_dataframes.append(pd.read_csv(file_info['csv'], low_memory=False))
            
            if dir_scenarios:
                # Use first scenario as representative (they should have same config)
                scenarios[name] = dir_scenarios[0]
                # Average dataframes
                dataframes[name] = average_dataframes_simple(dir_dataframes)
                print(f"✓ Loaded batch directory '{name}': averaged {len(dir_dataframes)} scenarios")
        else:
            # Single scenario - load normally
            if name in scenario_paths:
                paths = scenario_paths[name]
                with open(paths['pkl'], "rb") as f:
                    scenarios[name] = pickle.load(f)
                dataframes[name] = pd.read_csv(paths['csv'], low_memory=False)
            else:
                print(f"⚠️ Warning: Scenario '{name}' not found in scenario_paths")
    
    if not scenarios:
        return None
    
    # Get shared disaster info from first scenario
    first_scenario = list(scenarios.values())[0]
    disaster_start = first_scenario.desastre.start
    disaster_end = first_scenario.desastre.start + first_scenario.desastre.duration
    
    # Extract disaster node if it exists
    disaster_node = None
    if first_scenario.desastre.list_of_dict_node_per_start_time:
        disaster_node = first_scenario.desastre.list_of_dict_node_per_start_time[0]["node"]
    
    migration_times = {
        isp.isp_id: isp.datacenter.tempo_de_reacao
        for isp in first_scenario.lista_de_isps
        if isp.datacenter is not None
    }
    
    return {
        'scenarios': scenarios,
        'dataframes': dataframes,
        'disaster_start': disaster_start,
        'disaster_end': disaster_end,
        'disaster_node': disaster_node,
        'migration_times': migration_times,
    }

print("✓ Batch-aware load function ready!")


✓ Batch-aware load function ready!


In [4]:
# Helper function to add batch mode to GUI
def add_batch_mode_to_gui(gui_function_code):
    """
    This function will be used to patch the GUI creation function.
    For now, we'll modify the load_scenarios function directly in the GUI.
    """
    pass

# Note: Batch mode functionality is integrated into the GUI below
# Look for batch_mode_checkbox and the modified load_scenarios function


## 2. Interactive GUI

The GUI provides:
- Scenario selection dropdowns
- View mode toggle (Topology vs Statistics)
- Filtering options
- Visualization controls
- Timing marker toggles

In [5]:
def create_comparison_gui():
    """
    Create the main comparison GUI.
    """
    # Find available scenarios - scan both direct files and subdirectories
    output_path = Path("output")
    scenario_paths = {}  # Maps scenario_name -> {'pkl': path, 'csv': path}
    
    # Pattern 1: Direct files in output/ folder (old pattern)
    # Format: output/{scenario}.pkl and output/df_{scenario}.csv
    for pkl_file in output_path.glob("*.pkl"):
        scenario_name = pkl_file.stem
        csv_file = output_path / f"df_{scenario_name}.csv"
        if csv_file.exists():
            scenario_paths[scenario_name] = {
                'pkl': pkl_file,
                'csv': csv_file
            }
    
    # Pattern 2: Files in nested subdirectories (experiment pattern)
    # Format: output/{experiment}/{config}/scenario_{name}.pkl and dataframe_{name}.csv
    # Use rglob to recursively find all scenario_*.pkl files
    for pkl_file in output_path.rglob("scenario_*.pkl"):
        scenario_name = pkl_file.stem.replace("scenario_", "")
        csv_file = pkl_file.parent / f"dataframe_{scenario_name}.csv"
        if csv_file.exists():
            # Create a relative path name from output directory
            rel_path = pkl_file.parent.relative_to(output_path)
            full_name = f"{rel_path}/{scenario_name}".replace("\\", "/")
            scenario_paths[full_name] = {
                'pkl': pkl_file,
                'csv': csv_file
            }
    
    scenario_names = sorted(scenario_paths.keys())
    
    # Detect batch directories (directories with multiple scenarios)
    batch_directories = detect_batch_directories(output_path)
    
    # Create selection options: show directories for batch, individual scenarios for single
    # Initialize with default grouping behavior (will be rebuilt after toggle widget is created)
    selection_options = {}
    updated_display_names = {}
    
    # Add batch directories as options (default: grouped)
    for dir_path, files in batch_directories.items():
        display_name = dir_path.split('/')[-1] if '/' in dir_path else dir_path
        selection_options[dir_path] = {
            'type': 'batch',
            'files': files,
            'display_name': display_name
        }
        updated_display_names[dir_path] = display_name
    
    # Add single scenarios (not in batch directories) as individual options
    batch_scenario_paths = set()
    for dir_path, files in batch_directories.items():
        for f in files:
            batch_scenario_paths.add(f"{dir_path}/{f['name']}")
    
    for name in scenario_paths.keys():
        if name not in batch_scenario_paths:
            selection_options[name] = {
                'type': 'single',
                'files': [scenario_paths[name]],
                'display_name': name.split('/')[-1] if '/' in name else name
            }
            updated_display_names[name] = name.split('/')[-1] if '/' in name else name
    
    # Update scenario_names and scenario_display_names
    scenario_names = sorted(selection_options.keys())
    scenario_display_names = updated_display_names
    
    
    
    if len(scenario_names) < 1:
        print("Error: Need at least 1 valid scenario or directory in output/ folder")
        return
    
    # Create output widgets
    output_topology = widgets.Output()
    output_statistics = widgets.Output()
    output_comparison_table = widgets.Output()
    
    # Scenario selection
    # Store all available scenarios
    all_scenario_names = scenario_names.copy()
    
    # Directory grouping toggle
    directory_grouping_toggle = widgets.Checkbox(
        value=True,
        description='Group by Directory',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='180px')
    )
    
    # Scenario grouping toggle
    scenario_grouping_toggle = widgets.Checkbox(
        value=False,
        description='Group by Scenario',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='180px')
    )
    
    def rebuild_selection_options():
        """Rebuild selection options based on directory_grouping_toggle and scenario_grouping_toggle."""
        use_dir_grouping = directory_grouping_toggle.value
        use_scenario_grouping = scenario_grouping_toggle.value
        
        new_selection_options = {}
        new_updated_display_names = {}
        
        # If scenario grouping is ON, group by base scenario index across all directories
        if use_scenario_grouping:
            # Group scenarios by base index (e.g., base0, base1, etc.)
            scenario_groups = {}  # base_index -> list of file_info from all directories
            
            for dir_path, files in batch_directories.items():
                for file_info in files:
                    # Extract base index from file name
                    # file_info['name'] is pkl_file.stem.replace('scenario_', '')
                    # Format: "0.0_5.0_5.0_8" -> base index is "8" (last part after last underscore)
                    file_name = file_info['name']
                    if '_' in file_name:
                        parts = file_name.split('_')
                        if len(parts) >= 2:
                            base_index = parts[-1]  # Last part is the base index
                            if base_index not in scenario_groups:
                                scenario_groups[base_index] = []
                            scenario_groups[base_index].append({
                                'file_info': file_info,
                                'dir_path': dir_path
                            })
            
            # Create selection options for each scenario group
            for base_index, group_files in scenario_groups.items():
                group_name = f"scenario_base{base_index}"
                display_name = f"Base Scenario {base_index}"
                new_selection_options[group_name] = {
                    'type': 'batch',
                    'files': [item['file_info'] for item in group_files],
                    'display_name': display_name
                }
                new_updated_display_names[group_name] = display_name
            
            # Add single scenarios (not in batch directories)
            batch_scenario_paths = set()
            for dir_path, files in batch_directories.items():
                for f in files:
                    batch_scenario_paths.add(f"{dir_path}/{f['name']}")
            
            for name in scenario_paths.keys():
                if name not in batch_scenario_paths:
                    new_selection_options[name] = {
                        'type': 'single',
                        'files': [scenario_paths[name]],
                        'display_name': name.split('/')[-1] if '/' in name else name
                    }
                    new_updated_display_names[name] = name.split('/')[-1] if '/' in name else name
        
        elif use_dir_grouping:
            # ON: Show directories as single options (current behavior)
            # Add batch directories as options
            for dir_path, files in batch_directories.items():
                display_name = dir_path.split('/')[-1] if '/' in dir_path else dir_path
                new_selection_options[dir_path] = {
                    'type': 'batch',
                    'files': files,
                    'display_name': display_name
                }
                new_updated_display_names[dir_path] = display_name
            
            # Add single scenarios (not in batch directories) as individual options
            batch_scenario_paths = set()
            for dir_path, files in batch_directories.items():
                for f in files:
                    batch_scenario_paths.add(f"{dir_path}/{f['name']}")
            
            for name in scenario_paths.keys():
                if name not in batch_scenario_paths:
                    new_selection_options[name] = {
                        'type': 'single',
                        'files': [scenario_paths[name]],
                        'display_name': name.split('/')[-1] if '/' in name else name
                    }
                    new_updated_display_names[name] = name.split('/')[-1] if '/' in name else name
        else:
            # OFF: Show all individual scenarios from batch directories
            # Add all individual scenarios from batch directories
            for dir_path, files in batch_directories.items():
                dir_display_name = dir_path.split('/')[-1] if '/' in dir_path else dir_path
                for idx, file_info in enumerate(files):
                    individual_name = f"{dir_path}/{file_info['name']}"
                    display_name = f"{dir_display_name} (scenario {idx})"
                    new_selection_options[individual_name] = {
                        'type': 'single',
                        'files': [file_info],
                        'display_name': display_name
                    }
                    new_updated_display_names[individual_name] = display_name
            
            # Add single scenarios (not in batch directories)
            batch_scenario_paths = set()
            for dir_path, files in batch_directories.items():
                for f in files:
                    batch_scenario_paths.add(f"{dir_path}/{f['name']}")
            
            for name in scenario_paths.keys():
                if name not in batch_scenario_paths:
                    new_selection_options[name] = {
                        'type': 'single',
                        'files': [scenario_paths[name]],
                        'display_name': name.split('/')[-1] if '/' in name else name
                    }
                    new_updated_display_names[name] = name.split('/')[-1] if '/' in name else name
        
        new_scenario_names = sorted(new_selection_options.keys())
        return new_selection_options, new_scenario_names, new_updated_display_names
    
    # Scenario selection with multi-select
    scenario_select = widgets.SelectMultiple(
        options=[(scenario_display_names[name], name) for name in scenario_names],
        value=[scenario_names[0]] if scenario_names else [],
        description='Select Scenarios/Directories:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='400px', height='200px')
    )
    
    select_all_btn = widgets.Button(
        description='Select All',
        button_style='info',
        layout=widgets.Layout(width='100px')
    )
    
    clear_all_btn = widgets.Button(
        description='Clear All',
        button_style='warning',
        layout=widgets.Layout(width='100px')
    )
    
    def on_grouping_toggle_change(change):
        """Handle grouping toggle change (directory or scenario)."""
        # Prevent recursive updates
        if update_lock['locked']:
            return
        
        update_lock['locked'] = True
        try:
            # If scenario grouping is turned ON, turn off directory grouping (mutually exclusive)
            if change['owner'] == scenario_grouping_toggle and change['new']:
                directory_grouping_toggle.unobserve(on_directory_grouping_toggle_change, names='value')
                directory_grouping_toggle.value = False
                directory_grouping_toggle.observe(on_directory_grouping_toggle_change, names='value')
            # If directory grouping is turned ON, turn off scenario grouping (mutually exclusive)
            elif change['owner'] == directory_grouping_toggle and change['new']:
                scenario_grouping_toggle.unobserve(on_grouping_toggle_change, names='value')
                scenario_grouping_toggle.value = False
                scenario_grouping_toggle.observe(on_grouping_toggle_change, names='value')
            
            # Rebuild selection options
            nonlocal selection_options, scenario_names, scenario_display_names
            selection_options, scenario_names, scenario_display_names = rebuild_selection_options()
            
            # Update scenario_select widget
            scenario_select.options = [(scenario_display_names[name], name) for name in scenario_names]
            
            # Clear current selection and caches
            scenario_select.value = ()
            cache['plot_cache'].clear()
            cache['data'] = None
            
            # Update all_scenario_names for select_all button
            nonlocal all_scenario_names
            all_scenario_names = scenario_names.copy()
        finally:
            update_lock['locked'] = False
    
    def on_directory_grouping_toggle_change(change):
        """Handle directory grouping toggle change."""
        on_grouping_toggle_change(change)
    
    def on_select_all(b):
        scenario_select.value = tuple(scenario_names)
    
    def on_clear_all(b):
        scenario_select.value = ()
    
    select_all_btn.on_click(on_select_all)
    clear_all_btn.on_click(on_clear_all)
    directory_grouping_toggle.observe(on_directory_grouping_toggle_change, names='value')
    scenario_grouping_toggle.observe(on_grouping_toggle_change, names='value')
    
    # View mode
    view_mode = widgets.ToggleButtons(
        options=['Topology View', 'Statistics View', 'Comparison Table'],
        value='Statistics View',
        description='View:',
        button_style='info',
        style={'description_width': '50px'},
        layout=widgets.Layout(width='500px')
    )
    
    # Topology controls
    topology_scenario_dropdown = widgets.Dropdown(
        options=[],
        description='Show Topology:',
        style={'description_width': '100px'},
        layout=widgets.Layout(width='300px')
    )
    
    isp_dropdown = widgets.Dropdown(
        options=[],
        description='Select ISP:',
        style={'description_width': '80px'},
        layout=widgets.Layout(width='200px')
    )
    
    disaster_mode_checkbox = widgets.Checkbox(
        value=True,
        description='Disaster Mode (Remove Disaster Node)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='350px')
    )
    
    # Statistics controls
    filter_dropdown = widgets.Dropdown(
        options=['No Filter', 'Migration Traffic Only', 'Exclude Migration Traffic', 'By ISP', 'By Node'],
        value='No Filter',
        description='Filter:',
        style={'description_width': '60px'},
        layout=widgets.Layout(width='300px')
    )
    
    # Filter-specific controls
    filter_isp_dropdown = widgets.Dropdown(
        options=[],
        description='ISP:',
        style={'description_width': '40px'},
        layout=widgets.Layout(width='150px', display='none')
    )
    
    filter_node_dropdown = widgets.Dropdown(
        options=[],
        description='Node:',
        style={'description_width': '40px'},
        layout=widgets.Layout(width='150px', display='none')
    )
    
    viz_type_dropdown = widgets.Dropdown(
        options=['Blocking Probability Over Time', 'Availability Per Time Bucket', 
                 'Link Utilization', 'Network Usage'],
        value='Blocking Probability Over Time',
        description='Visualization:',
        style={'description_width': '90px'},
        layout=widgets.Layout(width='400px')
    )
    
    bucket_slider = widgets.IntSlider(
        value=10,
        min=1,
        max=50,
        step=1,
        description='Time Bucket:',
        style={'description_width': '90px'},
        layout=widgets.Layout(width='400px')
    )
    
    top_links_slider = widgets.IntSlider(
        value=15,
        min=5,
        max=50,  # Will be updated dynamically based on actual link count
        step=1,
        description='Top Links:',
        style={'description_width': '90px'},
        layout=widgets.Layout(width='400px', display='none')
    )
    
    show_migration_checkbox = widgets.Checkbox(
        value=True,
        description='Show ISP Migration Start Times',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    show_disaster_checkbox = widgets.Checkbox(
        value=True,
        description='Show Disaster Period',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='200px')
    )
    
    batch_view_toggle = widgets.Checkbox(
        value=True,
        description='Batch View',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='150px')
    )
    
    # Comparison Table controls
    table_filter_dropdown = widgets.Dropdown(
        options=['No Filter', 'Migration Traffic Only', 'Exclude Migration Traffic'],
        value='No Filter',
        description='Filter:',
        style={'description_width': '60px'},
        layout=widgets.Layout(width='300px')
    )
    
    status_label = widgets.HTML(
        value="",
        layout=widgets.Layout(margin='10px 0')
    )
    
    # Cache for loaded data and computed plots
    cache = {
        'data': None, 
        'scenario1_name': None, 
        'scenario2_name': None,
        'plot_cache': {},  # Cache for computed plots
        'last_plot_params': None  # Parameters of last plot
    }
    
    # Lock to prevent recursive callback loops
    update_lock = {'locked': False}
    
    # Update functions
    def load_scenarios():
        """Load selected scenarios."""
        # Ensure variables are accessible (they're in outer scope via closure)
        nonlocal batch_directories, scenario_paths, selection_options
        
        selected_names = list(scenario_select.value)
        
        if not selected_names:
            status_label.value = "<p style='color: orange;'>⚠️ Please select at least one scenario</p>"
            return None
        
        # Check cache
        cache_key = frozenset(selected_names)
        if cache.get('cache_key') == cache_key and cache['data'] is not None:
            return cache['data']
        
        # Load scenarios
        status_label.value = f"<p style='color: blue;'>🔄 Loading {len(selected_names)} scenarios...</p>"
        
        try:
            scenarios = {}
            dataframes = {}
            
            for name in selected_names:
                # Check if this is a batch directory or individual scenario
                # First check selection_options (which is what we use for the dropdown)
                # Then check batch_directories directly
                is_batch = False
                batch_files = None
                
                if name in selection_options:
                    option_type = selection_options[name].get('type')
                    if option_type == 'batch':
                        is_batch = True
                        batch_files = selection_options[name]['files']
                    elif option_type == 'single':
                        # Individual scenario from selection_options
                        file_info = selection_options[name]['files'][0]
                        # Load single scenario
                        with open(file_info['pkl'], "rb") as f:
                            scenarios[name] = pickle.load(f)
                        dataframes[name] = pd.read_csv(file_info['csv'], low_memory=False)
                        continue
                elif name in batch_directories:
                    is_batch = True
                    batch_files = batch_directories[name]
                
                if is_batch and batch_files:
                    # Load and average all scenarios in this directory
                    dir_scenarios = []
                    dir_dataframes = []
                    
                    for file_info in batch_files:
                        with open(file_info['pkl'], "rb") as f:
                            dir_scenarios.append(pickle.load(f))
                        dir_dataframes.append(pd.read_csv(file_info['csv'], low_memory=False))
                    
                    if dir_scenarios:
                        # Use first scenario as representative (they should have same config)
                        scenarios[name] = dir_scenarios[0]
                        # Average dataframes
                        dataframes[name] = average_dataframes_simple(dir_dataframes)
                        status_label.value += f"<br>✓ Averaged {len(dir_dataframes)} scenarios from directory '{name}'"
                elif name in scenario_paths:
                    # Single scenario - load normally
                    paths = scenario_paths[name]
                    with open(paths['pkl'], "rb") as f:
                        scenarios[name] = pickle.load(f)
                    dataframes[name] = pd.read_csv(paths['csv'], low_memory=False)
                else:
                    # Debug: show what keys are available
                    batch_keys = list(batch_directories.keys())[:5] if batch_directories else []
                    scenario_keys = list(scenario_paths.keys())[:5] if scenario_paths else []
                    error_msg = f"⚠️ Warning: '{name}' not found in scenario_paths or batch_directories"
                    error_msg += f"<br>Available batch directories (first 5): {batch_keys}"
                    error_msg += f"<br>Available scenario paths (first 5): {scenario_keys}"
                    status_label.value = f"<p style='color: red;'>{error_msg}</p>"
                    print(f"⚠️ Warning: '{name}' not found")
                    print(f"  Batch directories available: {list(batch_directories.keys())[:10]}")
                    print(f"  Scenario paths available: {list(scenario_paths.keys())[:10]}")
            
            # Get shared disaster info from first scenario
            first_scenario = list(scenarios.values())[0]
            disaster_start = first_scenario.desastre.start
            disaster_end = first_scenario.desastre.start + first_scenario.desastre.duration
            
            # Extract disaster node if it exists
            disaster_node = None
            if first_scenario.desastre.list_of_dict_node_per_start_time:
                disaster_node = first_scenario.desastre.list_of_dict_node_per_start_time[0]["node"]
            
            migration_times = {
                isp.isp_id: isp.datacenter.tempo_de_reacao
                for isp in first_scenario.lista_de_isps
                if isp.datacenter is not None
            }
            
            # Track which scenarios come from batch directories
            batch_scenario_info = {}
            for name in selected_names:
                if name in selection_options:
                    option_type = selection_options[name].get('type')
                    if option_type == 'batch':
                        batch_scenario_info[name] = {
                            'is_batch': True,
                            'files': selection_options[name]['files']
                        }
                    else:
                        # Individual scenario - check if it's from a batch directory
                        # Extract directory path from name (format: "dir_path/file_name")
                        if '/' in name:
                            dir_path = '/'.join(name.split('/')[:-1])
                            if dir_path in batch_directories:
                                # This individual scenario is from a batch directory
                                batch_scenario_info[name] = {
                                    'is_batch': False,  # It's an individual scenario
                                    'files': [selection_options[name]['files'][0]]  # Single file
                                }
                            else:
                                batch_scenario_info[name] = {
                                    'is_batch': False,
                                    'files': None
                                }
                        else:
                            batch_scenario_info[name] = {
                                'is_batch': False,
                                'files': None
                            }
                elif name in batch_directories:
                    batch_scenario_info[name] = {
                        'is_batch': True,
                        'files': batch_directories[name]
                    }
                else:
                    batch_scenario_info[name] = {
                        'is_batch': False,
                        'files': None
                    }
            
            data = {
                'scenarios': scenarios,
                'dataframes': dataframes,
                'disaster_start': disaster_start,
                'disaster_end': disaster_end,
                'disaster_node': disaster_node,
                'migration_times': migration_times,
                'batch_scenario_info': batch_scenario_info,  # Track batch directory info
            }
            
            cache['data'] = data
            cache['cache_key'] = cache_key
            
            # Update topology dropdown with display names
            display_names_list = [scenario_display_names[name] for name in selected_names]
            topology_scenario_dropdown.options = display_names_list
            if display_names_list:
                topology_scenario_dropdown.value = display_names_list[0]
            
            # Update ISP and filter dropdowns (use first scenario)
            isp_ids = [isp.isp_id for isp in first_scenario.lista_de_isps]
            isp_dropdown.options = isp_ids
            filter_isp_dropdown.options = isp_ids
            if isp_ids:
                isp_dropdown.value = isp_ids[0]
                filter_isp_dropdown.value = isp_ids[0]
            
            # Get nodes from first dataframe (any selected scenario/directory)
            first_df_key = list(dataframes.keys())[0]
            all_nodes = sorted(set(dataframes[first_df_key]['src'].unique()) | 
                              set(dataframes[first_df_key]['dst'].unique()))
            filter_node_dropdown.options = all_nodes
            if all_nodes:
                filter_node_dropdown.value = all_nodes[0]
            
            status_label.value = f"<p style='color: green;'>✅ Loaded {len(selected_names)} scenarios!</p>"
            return data
            
        except Exception as e:
            status_label.value = f"<p style='color: red;'>❌ Error: {str(e)}</p>"
            import traceback
            traceback.print_exc()
            return None
    
    def update_topology_view():
        """Update topology visualization."""
        with output_topology:
            clear_output(wait=True)
            
            data = load_scenarios()
            if data is None:
                return
            
            # Use selected topology scenario (convert display name back to full path)
            topology_display_name = topology_scenario_dropdown.value
            
            # Find the full path from display name
            topology_scenario_name = None
            for full_path, display_name in scenario_display_names.items():
                if display_name == topology_display_name and full_path in data['scenarios']:
                    topology_scenario_name = full_path
                    break
            
            if not topology_scenario_name:
                print("Please select a scenario to display topology")
                return
            
            scenario = data['scenarios'][topology_scenario_name]
            isp_id = isp_dropdown.value
            remove_disaster = disaster_mode_checkbox.value
            
            # Find ISP
            isp = next((isp for isp in scenario.lista_de_isps if isp.isp_id == isp_id), None)
            if isp is None:
                print(f"ISP {isp_id} not found")
                return
            
            print(f"⏳ Rendering topology for {topology_display_name}, ISP {isp_id}...")
            
            # Extract disaster node
            disaster_node = None
            if scenario.desastre.list_of_dict_node_per_start_time:
                disaster_node = scenario.desastre.list_of_dict_node_per_start_time[0]["node"]
            
            # Use existing single-scenario topology plot
            fig = visualize_isp_topology(
                isp,
                scenario.topology,
                disaster_node,
                topology_display_name,
                remove_disaster
            )
            
            clear_output(wait=True)
            if fig is not None:
                plt.show()
    
    def update_statistics_view():
        """Update statistics visualization."""
        with output_statistics:
            clear_output(wait=True)
            
            data = load_scenarios()
            if data is None:
                return
            
            # Check batch view toggle
            use_batch_view = batch_view_toggle.value
            
            # Process scenarios based on batch view toggle
            scenarios_to_use = {}
            dataframes_to_use = {}
            display_names_to_use = {}
            
            if use_batch_view:
                # Batch view ON: Show only directories (averaged scenarios)
                # Filter to keep only batch directories
                for name, scenario in data['scenarios'].items():
                    batch_info = data.get('batch_scenario_info', {}).get(name, {})
                    if batch_info.get('is_batch', False):
                        scenarios_to_use[name] = scenario
                        dataframes_to_use[name] = data['dataframes'][name]
                        display_names_to_use[name] = scenario_display_names[name]
            else:
                # Batch view OFF: Show individual scenarios
                # For batch directories, load individual scenarios
                # For non-batch, show as-is
                for name, scenario in data['scenarios'].items():
                    batch_info = data.get('batch_scenario_info', {}).get(name, {})
                    if batch_info.get('is_batch', False):
                        # Load individual scenarios from batch directory
                        batch_files = batch_info.get('files', [])
                        for idx, file_info in enumerate(batch_files):
                            try:
                                with open(file_info['pkl'], "rb") as f:
                                    individual_scenario = pickle.load(f)
                                individual_df = pd.read_csv(file_info['csv'], low_memory=False)
                                
                                # Create unique name for individual scenario
                                # Extract base name from file (e.g., "0.0_0.0_0.0_base0" -> "0.0_0.0_0.0_base0")
                                file_name = file_info['name']
                                individual_name = f"{name}/{file_name}"
                                
                                scenarios_to_use[individual_name] = individual_scenario
                                dataframes_to_use[individual_name] = individual_df
                                # Create display name: extract weights from file name
                                # file_info['name'] format: "0.0_5.0_5.0_8" where last part is base index
                                file_name = file_info['name']
                                if '_' in file_name:
                                    parts = file_name.split('_')
                                    if len(parts) >= 4:
                                        # Last part is base index, rest are weights (alpha_beta_gamma)
                                        weights = '_'.join(parts[:-1])  # e.g., "0.0_5.0_5.0"
                                        # Use weights in display name
                                        display_names_to_use[individual_name] = f"{scenario_display_names[name]} ({weights})"
                                    else:
                                        display_names_to_use[individual_name] = f"{scenario_display_names[name]} (scenario {idx})"
                                else:
                                    display_names_to_use[individual_name] = f"{scenario_display_names[name]} (scenario {idx})"
                            except Exception as e:
                                print(f"Warning: Could not load individual scenario {file_info['name']}: {e}")
                    else:
                        # Non-batch scenario: show as-is
                        scenarios_to_use[name] = scenario
                        dataframes_to_use[name] = data['dataframes'][name]
                        display_names_to_use[name] = scenario_display_names[name]
            
            if not scenarios_to_use:
                print("No scenarios to display")
                return
            
            # Extract disaster and migration info from scenarios_to_use
            # Use first scenario (all should have same disaster/migration info if from same batch)
            first_scenario_to_use = list(scenarios_to_use.values())[0]
            disaster_start = first_scenario_to_use.desastre.start
            disaster_end = first_scenario_to_use.desastre.start + first_scenario_to_use.desastre.duration
            migration_times = {
                isp.isp_id: isp.datacenter.tempo_de_reacao
                for isp in first_scenario_to_use.lista_de_isps
                if isp.datacenter is not None
            }
            
            # Apply filters to all dataframes
            filter_type = filter_dropdown.value
            filter_kwargs = {}
            
            if filter_type == 'By ISP':
                filter_kwargs['isp_id'] = filter_isp_dropdown.value
            elif filter_type == 'By Node':
                filter_kwargs['node'] = filter_node_dropdown.value
            
            filtered_dfs = {
                name: apply_filter(df, filter_type, **filter_kwargs)
                for name, df in dataframes_to_use.items()
            }
            
            # Check if any data available
            if all(len(df) == 0 for df in filtered_dfs.values()):
                print("No data available after applying filter")
                return
            
            # Get parameters
            viz_type = viz_type_dropdown.value
            bucket_size = bucket_slider.value
            top_n_links = top_links_slider.value
            show_migration = show_migration_checkbox.value
            show_disaster = show_disaster_checkbox.value
            
            # Create cache key (include batch_view_toggle)
            cache_key = (
                frozenset(scenarios_to_use.keys()),
                filter_type,
                filter_kwargs.get('isp_id'),
                filter_kwargs.get('node'),
                viz_type,
                bucket_size if viz_type in ['Blocking Probability Over Time', 'Availability Per Time Bucket'] else None,
                top_n_links if viz_type == 'Link Utilization' else None,
                show_migration,
                show_disaster,
                use_batch_view  # Include batch view toggle in cache key
            )
            
            if cache_key in cache['plot_cache']:
                fig = cache['plot_cache'][cache_key]
                display(fig)
                return
            
            print(f"⏳ Computing {viz_type}...")
            
            # Create display name mappings for plots
            filtered_dfs_display = {
                display_names_to_use[name]: df
                for name, df in filtered_dfs.items()
            }
            
            fig, ax = plt.subplots(figsize=(16, 8))
            
            # Use multi-scenario plot functions with display names
            if viz_type == 'Blocking Probability Over Time':
                plot_blocking_probability_multi(
                    ax, filtered_dfs_display, bucket_size,
                    disaster_start, disaster_end,
                    migration_times, show_migration, show_disaster
                )
            
            elif viz_type == 'Availability Per Time Bucket':
                plot_availability_multi(
                    ax, filtered_dfs_display, bucket_size,
                    disaster_start, disaster_end,
                    migration_times, show_migration, show_disaster
                )
            
            elif viz_type == 'Link Utilization':
                plot_link_utilization_multi(
                    ax, filtered_dfs_display, top_n=top_n_links
                )
            
            elif viz_type == 'Network Usage':
                first_scenario = list(scenarios_to_use.values())[0]
                plot_network_usage_multi(
                    ax, filtered_dfs_display, first_scenario.topology,
                    disaster_start, disaster_end,
                    migration_times, show_migration, show_disaster
                )
            
            plt.tight_layout()
            cache['plot_cache'][cache_key] = fig
            
            clear_output(wait=True)
            plt.show()
    
    def update_comparison_table():
        """Update comparison table view."""
        with output_comparison_table:
            clear_output(wait=True)
            
            data = load_scenarios()
            if data is None:
                return
            
            # Apply filter
            filter_type = table_filter_dropdown.value
            filtered_dfs = {
                name: apply_filter(df, filter_type)
                for name, df in data['dataframes'].items()
            }
            
            if all(len(df) == 0 for df in filtered_dfs.values()):
                print("No data available after applying filter")
                return
            
            print("⏳ Generating comparison table...")
            
            # Create display name mappings for table
            scenarios_display = {
                scenario_display_names[name]: scenario 
                for name, scenario in data['scenarios'].items()
            }
            filtered_dfs_display = {
                scenario_display_names[name]: df
                for name, df in filtered_dfs.items()
            }
            
            # Use multi-scenario comparison function with display names
            comparison_data = create_multi_scenario_comparison_data(
                scenarios_display,
                filtered_dfs_display,
                data['disaster_start'],
                data['disaster_end'],
            )
            
            clear_output(wait=True)
            
            # Display table
            table_widget = create_multi_scenario_table_widget(comparison_data)
            display(table_widget)
            
            # Display traffic info
            print(f"\nTraffic Filter: {filter_type}")
            for name, df in filtered_dfs.items():
                display_name = scenario_display_names[name]
                total_original = len(data['dataframes'][name])
                print(f"{display_name}: {len(df)} requests ({len(df)/total_original*100:.1f}% of total)")                    
    
    def update_view_for_current_mode():
        """Update the currently active view."""
        if view_mode.value == 'Topology View':
            update_topology_view()
        elif view_mode.value == 'Statistics View':
            update_statistics_view()
        elif view_mode.value == 'Comparison Table':
            update_comparison_table()
    
    def on_view_mode_change(change):
        """Handle view mode change."""
        if change['new'] == 'Topology View':
            output_topology.layout.display = 'block'
            output_statistics.layout.display = 'none'
            output_comparison_table.layout.display = 'none'
            topology_controls_row.layout.display = 'flex'
            statistics_controls_rows.layout.display = 'none'
            table_controls_row.layout.display = 'none'
            update_topology_view()
        elif change['new'] == 'Statistics View':
            output_topology.layout.display = 'none'
            output_statistics.layout.display = 'block'
            output_comparison_table.layout.display = 'none'
            topology_controls_row.layout.display = 'none'
            statistics_controls_rows.layout.display = 'flex'
            table_controls_row.layout.display = 'none'
            update_statistics_view()
        else:  # Comparison Table
            output_topology.layout.display = 'none'
            output_statistics.layout.display = 'none'
            output_comparison_table.layout.display = 'block'
            topology_controls_row.layout.display = 'none'
            statistics_controls_rows.layout.display = 'none'
            table_controls_row.layout.display = 'flex'
            update_comparison_table()
    
    def on_filter_change(change):
        """Handle filter type change."""
        filter_type = change['new']
        if filter_type == 'By ISP':
            filter_isp_dropdown.layout.display = 'block'
            filter_node_dropdown.layout.display = 'none'
        elif filter_type == 'By Node':
            filter_isp_dropdown.layout.display = 'none'
            filter_node_dropdown.layout.display = 'block'
        else:
            filter_isp_dropdown.layout.display = 'none'
            filter_node_dropdown.layout.display = 'none'
        
        if view_mode.value == 'Statistics View':
            update_statistics_view()
    
    def on_scenario_change(change):
        """Handle scenario selection change."""
        if update_lock['locked']:
            return
        
        update_lock['locked'] = True
        try:
            selected = list(scenario_select.value)
            
            # Validate at least one scenario
            if not selected:
                status_label.value = "<p style='color: orange;'>⚠️ Please select at least one scenario</p>"
                update_lock['locked'] = False
                return
            
            # Clear caches
            cache['plot_cache'].clear()
            cache['data'] = None
            
            # Update view
            update_view_for_current_mode()
        finally:
            update_lock['locked'] = False
    
    def on_topology_param_change(change):
        """Handle topology parameter change."""
        if view_mode.value == 'Topology View':
            update_topology_view()
    
    def on_viz_type_change(change):
        """Handle visualization type change and update slider visibility."""
        viz_type = change['new']
        
        # Show/hide appropriate sliders based on visualization type
        if viz_type in ['Blocking Probability Over Time', 'Availability Per Time Bucket']:
            bucket_slider.layout.display = 'block'
            top_links_slider.layout.display = 'none'
        elif viz_type == 'Link Utilization':
            bucket_slider.layout.display = 'none'
            top_links_slider.layout.display = 'block'
        else:  # Network Usage
            bucket_slider.layout.display = 'none'
            top_links_slider.layout.display = 'none'
        
        # Update view
        if view_mode.value == 'Statistics View':
            update_statistics_view()
    
    def on_batch_view_toggle_change(change):
        """Handle batch view toggle change."""
        if view_mode.value == 'Statistics View':
            # Clear plot cache when toggling batch view (data changes)
            cache['plot_cache'].clear()
            update_statistics_view()
    
    def on_statistics_param_change(change):
        """Handle statistics parameter change."""
        if view_mode.value == 'Statistics View':
            update_statistics_view()
    
    def on_table_param_change(change):
        """Handle comparison table parameter change."""
        if view_mode.value == 'Comparison Table':
            update_comparison_table()
    
    # Attach observers
    view_mode.observe(on_view_mode_change, names='value')
    scenario_select.observe(on_scenario_change, names='value')
    topology_scenario_dropdown.observe(on_topology_param_change, names='value')
    isp_dropdown.observe(on_topology_param_change, names='value')
    disaster_mode_checkbox.observe(on_topology_param_change, names='value')
    filter_dropdown.observe(on_filter_change, names='value')
    filter_isp_dropdown.observe(on_statistics_param_change, names='value')
    filter_node_dropdown.observe(on_statistics_param_change, names='value')
    viz_type_dropdown.observe(on_viz_type_change, names='value')
    bucket_slider.observe(on_statistics_param_change, names='value')
    top_links_slider.observe(on_statistics_param_change, names='value')
    show_migration_checkbox.observe(on_statistics_param_change, names='value')
    show_disaster_checkbox.observe(on_statistics_param_change, names='value')
    batch_view_toggle.observe(on_batch_view_toggle_change, names='value')
    table_filter_dropdown.observe(on_table_param_change, names='value')
    
    # Layout
    scenario_selection_box = widgets.VBox([
        widgets.HBox([select_all_btn, clear_all_btn, directory_grouping_toggle, scenario_grouping_toggle]),
        scenario_select
    ])
    selection_row = widgets.HBox([scenario_selection_box])
    view_row = widgets.HBox([view_mode])
    
    topology_controls_row = widgets.HBox(
        [topology_scenario_dropdown, isp_dropdown, disaster_mode_checkbox],
        layout=widgets.Layout(display='none', margin='5px 0')
    )
    
    filter_row = widgets.HBox([filter_dropdown, filter_isp_dropdown, filter_node_dropdown])
    viz_row = widgets.HBox([viz_type_dropdown])
    sliders_row = widgets.HBox([bucket_slider, top_links_slider])
    toggles_row = widgets.HBox([show_migration_checkbox, show_disaster_checkbox, batch_view_toggle])
    
    statistics_controls_rows = widgets.VBox(
        [filter_row, viz_row, sliders_row, toggles_row],
        layout=widgets.Layout(display='flex', margin='5px 0')
    )
    
    table_controls_row = widgets.HBox(
        [table_filter_dropdown],
        layout=widgets.Layout(display='none', margin='5px 0')
    )
    
    gui = widgets.VBox([
        selection_row,
        view_row,
        topology_controls_row,
        statistics_controls_rows,
        table_controls_row,
        status_label,
        output_topology,
        output_statistics,
        output_comparison_table
    ])
    
    # Initialize
    update_statistics_view()
    
    return gui

## 3. Launch GUI

In [6]:
# Launch the GUI
gui = create_comparison_gui()
display(gui)
